In [ ]:
# Colab dependency setup.
# Run this first in Google Colab, or use Runtime > Run all.
import sys

try:
    import google.colab  # type: ignore
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    get_ipython().system(f"{sys.executable} -m pip install -q pandas numpy scikit-learn matplotlib openai")


# Notebook 1 — Component-Level Evaluation
### Evaluating AI Agents: From Component Checks to Adversarial Defense

---

## Learning Objectives

By the end of this notebook you will be able to:

1. Run a mock research assistant agent over a component evaluation dataset.
2. Compute **tool-selection accuracy** — did the agent pick the right tool?
3. Compute **argument exact match** and **field-level F1** — did it use the right parameters?
4. Inspect failures by tool and failure category to identify systematic weaknesses.

> **Why component evaluation?**  
> Before measuring whether an agent gives a good final answer, you need to check whether its sub-decisions are correct.  
> A wrong tool selection early in a trajectory will corrupt every downstream step — no matter how good the final answer looks.

In [ ]:
#@title Setup and runtime options
import os
import sys
from pathlib import Path

#@markdown Check `USE_OPENAI` to call a small baseline OpenAI model. Add `OPENAI_API_KEY` in Colab Secrets first.
USE_OPENAI = False #@param {type:"boolean"}
OPENAI_MODEL = "gpt-4.1-nano" #@param {type:"string"}

REPO_URL = "https://github.com/AmmarMohanna/packt-ai-agents-eval.git"
REPO_NAME = "packt-ai-agents-eval"

try:
    import google.colab  # type: ignore
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    REPO_PATH = Path("/content") / REPO_NAME
    os.chdir("/content")
    get_ipython().system(f"rm -rf {REPO_PATH}")
    get_ipython().system(f"git clone -q {REPO_URL} {REPO_PATH}")
    PROJECT_ROOT = REPO_PATH
    if not (PROJECT_ROOT / "notebooks").is_dir():
        raise RuntimeError(f"Could not clone course repo into {PROJECT_ROOT}. Re-run this cell or restart the runtime.")
else:
    cwd = Path.cwd().resolve()
    candidates = [cwd, *cwd.parents]
    PROJECT_ROOT = next(
        (path for path in candidates if (path / "src").is_dir() and (path / "data").is_dir()),
        cwd,
    )

sys.path.insert(0, str(PROJECT_ROOT))
os.chdir(PROJECT_ROOT / "notebooks")

import importlib
importlib.invalidate_caches()
for module_name in list(sys.modules):
    if module_name == "src" or module_name.startswith("src."):
        sys.modules.pop(module_name, None)

OPENAI_API_KEY = None
if USE_OPENAI:
    if not IN_COLAB:
        raise RuntimeError("OpenAI mode is configured for Google Colab Secrets.")
    from google.colab import userdata
    OPENAI_API_KEY = userdata.get("OPENAI_API_KEY")
    if not OPENAI_API_KEY:
        raise RuntimeError("Add OPENAI_API_KEY in the Colab Secrets panel, then rerun the notebook.")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams["figure.dpi"] = 110

from src.trace_utils import load_jsonl
from src.mock_agent import run_component_prediction
from src.openai_agents import predict_component_with_openai
from src.metrics import (
    tool_selection_accuracy,
    argument_exact_match,
    argument_field_f1,
    component_failure_table,
)

pd.set_option("display.max_colwidth", 60)
print("Setup complete.")

In [ ]:
# ── Load the component evaluation dataset ─────────────────────────────────
DATA_PATH = os.path.join("..", "data", "component_dataset.jsonl")
examples = load_jsonl(DATA_PATH)

print(f"Loaded {len(examples)} evaluation examples.")
print("\nSample example:")
import json; print(json.dumps(examples[0], indent=2))

In [ ]:
# ── Run the selected agent on every query ──────────────────────────────────
#
# Default mode uses the deterministic mock router.
# If USE_OPENAI is checked above, the notebook uses OpenAI to choose tools and arguments.

FAILURE_INJECTION = {
    "cmp_003": "wrong_tool",   # Should search_web, but picks wrong tool
    "cmp_007": "bad_args",     # Should read_document with doc_id, but sends {}
    "cmp_011": "wrong_tool",   # Should summarize_evidence, picks wrong tool
}

records = []
for ex in examples:
    if USE_OPENAI:
        pred = predict_component_with_openai(ex, api_key=OPENAI_API_KEY, model=OPENAI_MODEL)
    else:
        profile = FAILURE_INJECTION.get(ex["id"])   # None for clean examples
        pred = run_component_prediction(ex["query"], profile=profile)

    records.append({
        "id":                   ex["id"],
        "query":                ex["query"],
        "expected_tool":        ex["expected_tool"],
        "predicted_tool":       pred["predicted_tool"],
        "expected_arguments":   ex["expected_arguments"],
        "predicted_arguments":  pred["predicted_arguments"],
        "failure_category":     ex.get("failure_category"),
    })

df = pd.DataFrame(records)
mode = f"OpenAI ({OPENAI_MODEL})" if USE_OPENAI else "deterministic mock router"
print(f"Mode: {mode}")
print(f"Predictions generated for {len(df)} examples.")


In [ ]:
# ── Compute tool-selection accuracy ───────────────────────────────────────
y_true = df["expected_tool"].tolist()
y_pred = df["predicted_tool"].tolist()

accuracy = tool_selection_accuracy(y_true, y_pred)
print(f"Tool-selection accuracy: {accuracy:.1%}  ({int(accuracy * len(y_true))}/{len(y_true)} correct)")

In [ ]:
# ── Compute argument exact match and field-level F1 ────────────────────────
#
# Argument metrics are only meaningful when the tool was correctly selected.
# For wrong-tool predictions, argument scores are 0 by definition.

failure_rows = component_failure_table(records)
df_eval = pd.DataFrame(failure_rows)

# Summary stats
correct_tool = df_eval["tool_correct"].sum()
arg_em = df_eval[df_eval["tool_correct"]]["arg_exact_match"].mean()
arg_f1 = df_eval[df_eval["tool_correct"]]["arg_field_f1"].mean()

print(f"Tool correct:            {correct_tool}/{len(df_eval)}")
print(f"Argument exact match:    {arg_em:.1%}  (among correct-tool examples)")
print(f"Argument field-level F1: {arg_f1:.3f}  (among correct-tool examples)")

In [ ]:
# ── Tool confusion matrix ──────────────────────────────────────────────────
tools = sorted(df_eval["expected_tool"].unique())
confusion = pd.crosstab(
    df_eval["expected_tool"],
    df_eval["predicted_tool"],
    rownames=["Expected"],
    colnames=["Predicted"],
)
# Reindex to ensure all tools appear even if no predictions
confusion = confusion.reindex(index=tools, columns=tools, fill_value=0)

fig, ax = plt.subplots(figsize=(7, 5))
im = ax.imshow(confusion.values, cmap="Blues")
ax.set_xticks(range(len(tools))); ax.set_xticklabels(tools, rotation=35, ha="right", fontsize=9)
ax.set_yticks(range(len(tools))); ax.set_yticklabels(tools, fontsize=9)
ax.set_xlabel("Predicted Tool"); ax.set_ylabel("Expected Tool")
ax.set_title("Tool Selection Confusion Matrix")
for i in range(len(tools)):
    for j in range(len(tools)):
        ax.text(j, i, confusion.values[i, j], ha="center", va="center", fontsize=11)
plt.colorbar(im, ax=ax)
plt.tight_layout()
plt.savefig(os.path.join("..", "outputs", "nb01_tool_confusion.png"), bbox_inches="tight")
plt.show()

In [ ]:
# ── Failure report: top failed examples ───────────────────────────────────
failed = df_eval[
    (~df_eval["tool_correct"]) | (~df_eval["arg_exact_match"])
].copy()

print(f"\n{'='*70}")
print(f"  FAILURE REPORT  ({len(failed)} examples with tool or argument issues)")
print(f"{'='*70}")

display_cols = ["id", "expected_tool", "predicted_tool", "tool_correct",
                "arg_exact_match", "arg_field_f1", "failure_category"]
print(failed[display_cols].to_string(index=False))

# Per-tool failure breakdown
print("\nFailures by expected tool:")
print(
    df_eval.groupby("expected_tool")[["tool_correct", "arg_exact_match"]]
    .mean().round(3).to_string()
)

In [ ]:
# ── Full results table ─────────────────────────────────────────────────────
print("Full component evaluation results:")
print(df_eval[display_cols].to_string(index=False))

## Interpreting the Results

| Metric | What it tells you |
|--------|-------------------|
| **Tool-selection accuracy** | Are the agent's routing decisions correct? A score below ~90% means the agent is regularly trying to use the wrong tool for the job. |
| **Argument exact match** | Is every argument exactly right? This is strict — even a minor argument difference counts as a miss. |
| **Argument field-level F1** | More forgiving — partial credit for fields that *were* correct. Useful when arguments have many optional fields. |

**Key insight:** The confusion matrix often reveals *systematic* routing errors.  
For example, if `search_web` is frequently confused with `search_docs`, the agent lacks a clear signal for when to use internal vs. external retrieval.

**What to do with failures:**
- `wrong_tool` errors → improve routing logic or add examples to the routing classifier.
- `bad_args` errors → improve argument extraction / schema validation at the tool call boundary.
- `ambiguous_query` failures → verify that the agent correctly escalates to `ask_clarification` instead of guessing.

## Try It Yourself

**Extension exercise** — Add a new evaluation example where the query is underspecified and the correct response is to ask for clarification:

```python
new_example = {
    "id": "cmp_021",
    "query": "Tell me about the thing.",
    "expected_tool": "ask_clarification",
    "expected_arguments": {"question": "Could you clarify: Tell me about the thing.?"},
    "allowed_tools": ["ask_clarification"],
    "failure_category": "ambiguous_query",
}

pred = run_component_prediction(new_example["query"])
print("Predicted tool:", pred["predicted_tool"])
print("Expected tool: ", new_example["expected_tool"])
print("Tool correct:  ", pred["predicted_tool"] == new_example["expected_tool"])
```

**Questions to explore:**
- What query patterns does the agent *fail* to route to `ask_clarification`?
- What would you need to add to `_pick_tool_and_args()` in `src/mock_agent.py` to improve coverage?
- How would you automate regression testing for the routing logic?